In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
customers_bronze = spark.table('''
                               quickcart.bronze.customers
                               ''')
customers_bronze.limit(5).display()

###Standardize the columns

In [0]:
customers_standardized = customers_bronze\
    .withColumn("customer_id", F.trim(F.col("customer_id")))\
        .withColumn("customer_name", F.trim(F.col("customer_name")))\
            .withColumn("email", F.lower(F.trim(F.col("email"))))\
                .withColumn("phone", F.regexp_replace(F.trim(F.col("phone")),r"\s+",""))\
                    .withColumn("gender", F.upper(F.trim(F.col("gender"))))\
                        .withColumn("city", F.initcap(F.trim(F.col("city"))))\
                            .withColumn("state", F.initcap(F.trim(F.col("state"))))\
                                .withColumn("pincode", F.trim(F.col("pincode")))\
                                    .withColumn("customer_segment", F.upper(F.trim(F.col("customer_name"))))

customers_standardized.limit(5).display()

###Create validation flags

In [0]:
customers_validated = customers_standardized\
    .withColumn("invalid_customer_id", F.col("customer_id").isNull() | (F.col("customer_id") == ""))\
        .withColumn("invalid_email", (F.col("email").isNotNull() & ~F.col("email").rlike(
                r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            )))\
                .withColumn("invalid_gender", ( F.col("gender").isNotNull()
            &
            ~F.col("gender").isin(
                "M",
                "F",
                "MALE",
                "FEMALE",
                "OTHER")))

customers_validated = customers_validated\
    .withColumn("record_status",
                F.when(F.col("invalid_customer_id"),"INVALID_CUSTOMER_ID")\
                    .when(F.col("invalid_email"),"INVALID_EMAIL")\
                        .when(F.col("invalid_gender"),"INVALID_GENDER")\
                            .otherwise("VALID")
                )
    
display(customers_validated.groupBy("invalid_gender").agg(count("invalid_gender")).alias("c"))


###Separate valid and invalid records

In [0]:
customers_valid = customers_validated\
    .filter(col("record_status") == "VALID")

customers_invalid = customers_validated\
    .filter(col("record_status") != "VALID") 

In [0]:
print(
    "Valid records:",
    customers_valid.count()
)

print(
    "Invalid records:",
    customers_invalid.count()
)

###Handle duplicates

In [0]:
customer_window = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())

In [0]:
customer_deduplicated = customers_valid\
    .withColumn("_row_number", F.row_number().over(customer_window))\
        .filter(col("_row_number") == 1)\
            .drop("_row_number")

customer_deduplicated.groupBy("customer_id").count().filter(F.col("count")>1).display()

###Remove technical validation columns

In [0]:
customers_silver = customer_deduplicated\
    .drop("invalid_customer_id","invalid_email","invalid_gender","record_status")

###Add Silver audit columns

In [0]:
customers_silver = customers_silver\
    .withColumn("_silver_processed_timestamp",F.current_timestamp())\
        .withColumn("_silver_source",F.lit("quickcart.bronze.customers"))

display(customers_silver.limit(5))

###Write the Silver table

In [0]:
customers_silver.write.format("delta").mode("overwrite").saveAsTable("quickcart.silver.customers")

In [0]:
%sql
SELECT COUNT(*) AS silver_count FROM quickcart.silver.customers;

SELECT *
FROM quickcart.silver.customers
LIMIT 5;

###Create the quarantine table

In [0]:
customers_invalid = (
    customers_invalid

    .withColumn(
        "_quarantine_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_quarantine_source",
        F.lit("quickcart.bronze.customers")
    )
)

In [0]:
(
    customers_invalid
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "quickcart.quarantine.customers"
    )
)

In [0]:
%sql
SELECT
    record_status,
    COUNT(*) AS record_count
FROM quickcart.quarantine.customers
GROUP BY record_status;